In [1]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, time
import numpy as np
from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings("ignore")
import glob

/home/cloudcraftz/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
underlying = 'ASIANPAINT'
report_path = f'/home/cloudcraftz/Downloads/data_reports/2024/{underlying}.csv'

In [3]:
df = pd.read_csv(report_path, parse_dates=["Date"], index_col="Date")

In [27]:
def get_title_liq(text, underlying):
    moneyness_pct = int(text.split("_")[-1])
    opt_type = text.split("_")[-2]

    if moneyness_pct == 0:
        return f"ATM {opt_type} | {underlying}"

    elif moneyness_pct < 0:
        return f"{moneyness_pct}% ITM {opt_type} | {underlying}"

    elif moneyness_pct > 0:
        return f"{moneyness_pct}% OTM {opt_type} | {underlying}"

def liquidity_plot(col_name, title = '',bins = 50, underlying=underlying):

    liq_values = df[col_name].dropna()
    mean = liq_values.mean()
    sigma = liq_values.std()

    # Compute Kernel Density Estimate (KDE)
    kde = gaussian_kde(liq_values)
    x_vals = np.linspace(liq_values.min(), liq_values.max(), 200)
    kde_vals = kde(x_vals)

    # Create histogram and density plot
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=liq_values,
        histnorm='probability density',
        nbinsx=bins,
        name='Liquidity Density',
        opacity=0.6,
        marker=dict(color='blue')
    ))

    # Add KDE line
    # fig.add_trace(go.Scatter(x=x_vals, y=kde_vals, mode='lines', name='KDE', line=dict(color='orange')))

    # Add mean and ±2*sigma bands as vertical lines
    fig.add_trace(go.Scatter(x=[mean, mean], y=[0, max(kde_vals)], mode='lines', name='Mean', line=dict(color='red', dash='dash')))
    fig.add_trace(go.Scatter(x=[mean - 2 * sigma, mean - 2 * sigma], y=[0, max(kde_vals)], mode='lines', name='Mean - 2σ', line=dict(color='green', dash='dot')))
    fig.add_trace(go.Scatter(x=[mean + 2 * sigma, mean + 2 * sigma], y=[0, max(kde_vals)], mode='lines', name='Mean + 2σ', line=dict(color='green', dash='dot')))

    title2 = get_title_liq(text=col_name, underlying=underlying)

    # Update layout
    fig.update_layout(
        title=f'{title} | {title2}',
        xaxis_title='Liquidity (%)',
        yaxis_title='Density',
        bargap=0.05,
        template='plotly_white'
    )

    fig.show()

def avg_spread_plot(col_name, title = '',bins = 50):

    liq_values = df[col_name].dropna()
    mean = liq_values.mean()
    sigma = liq_values.std()

    # Compute Kernel Density Estimate (KDE)
    kde = gaussian_kde(liq_values)
    x_vals = np.linspace(liq_values.min(), liq_values.max(), 200)
    kde_vals = kde(x_vals)

    # Create histogram and density plot
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=liq_values,
        histnorm='probability density',
        nbinsx=bins,
        name='Avg. Spread Density',
        opacity=0.6,
        marker=dict(color='blue')
    ))

    # Add KDE line
    fig.add_trace(go.Scatter(x=x_vals, y=kde_vals, mode='lines', name='KDE', line=dict(color='orange')))

    # Add mean and ±2*sigma bands as vertical lines
    fig.add_trace(go.Scatter(x=[mean, mean], y=[0, max(kde_vals)], mode='lines', name='Mean', line=dict(color='red', dash='dash')))
    fig.add_trace(go.Scatter(x=[mean - 2 * sigma, mean - 2 * sigma], y=[0, max(kde_vals)], mode='lines', name='Mean - 2σ', line=dict(color='green', dash='dot')))
    fig.add_trace(go.Scatter(x=[mean + 2 * sigma, mean + 2 * sigma], y=[0, max(kde_vals)], mode='lines', name='Mean + 2σ', line=dict(color='green', dash='dot')))

    # Update layout
    fig.update_layout(
        title=f'{title} | {col_name}',
        xaxis_title='Avg. Spread (%)',
        yaxis_title='Density',
        bargap=0.05,
        template='plotly_white'
    )

    fig.show()

def spread_time_plot(col_name, title = ''):

    # Count occurrences
    time_counts = df[col_name].value_counts().sort_index()

    # Create line chart
    fig_time = go.Figure()
    fig_time.add_trace(go.Scatter(
        x=time_counts.index,
        y=time_counts.values,
        mode='lines+markers',
        name='Time Count',
        line=dict(color='blue')
    ))

    # Update layout
    fig_time.update_layout(
        title=f'{title}',
        xaxis_title='Time',
        yaxis_title='Count',
        template='plotly_white'
    )

    fig_time.show()

In [39]:
def liquidity_plot2(col_name, title='', bins=50, underlying=underlying):
    liq_values = df[col_name].dropna()
    mean = liq_values.mean()
    sigma = liq_values.std()

    # Use numpy to bucket values
    counts, bin_edges = np.histogram(liq_values, bins=bins)  # density=True normalizes area under bar chart to 1
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  # Midpoints of bins for x-axis
    tot_count = counts.sum()
    count_percentage = counts/tot_count

    # Create bar chart instead of histogram
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=bin_centers,
        y=count_percentage,
        name='Liquidity Density',
        marker=dict(color='blue'),
        opacity=0.6
    ))

    # KDE (optional)
    kde = gaussian_kde(liq_values)
    x_vals = np.linspace(liq_values.min(), liq_values.max(), 200)
    kde_vals = kde(x_vals)
    fig.add_trace(go.Scatter(x=x_vals, y=kde_vals, mode='lines', name='KDE', line=dict(color='orange')))

    # Add vertical lines for mean and ±2σ
    fig.add_trace(go.Scatter(x=[mean, mean], y=[0, max(kde_vals)], mode='lines', name='Mean', line=dict(color='red', dash='dash')))

    fig.add_trace(go.Scatter(x=[mean - 2 * sigma, mean - 2 * sigma], y=[0, max(kde_vals)], mode='lines',
                             name='Mean - 2σ', line=dict(color='green', dash='dot')))
    fig.add_trace(go.Scatter(x=[mean + 2 * sigma, mean + 2 * sigma], y=[0, max(kde_vals)], mode='lines',
                             name='Mean + 2σ', line=dict(color='green', dash='dot')))

    # Format title
    title2 = get_title_liq(text=col_name, underlying=underlying)

    fig.update_layout(
        title=f'{title} | {title2}',
        xaxis_title='Liquidity (%)',
        yaxis_title='No of Days (%)',
        bargap=0.05,
        template='plotly_white'
    )

    fig.show()

col_name_list = ['liq_CE_0', 'liq_PE_0', 'liq_CE_5', 'liq_PE_5', 'liq_PE_10', 'liq_CE_10']

for col_name in col_name_list:
    liquidity_plot2(col_name, title='Liquidity Plot')


In [57]:
import pandas as pd
import numpy as np

def liquidity_bucket_table(df, step=10, min_val=1, max_val=100):
    # Define bin edges and labels
    bins = list(range(min_val, max_val + 1, step))
    bins.append(max_val + 1)  # to include the last value
    labels = [f"{i}-{i+step-1}" for i in range(min_val, max_val + 1, step)]

    # Initialize result DataFrame
    result = pd.DataFrame(index=df.columns, columns=labels)

    # Count values in each bin for each column
    for col in df.columns:
        counts = pd.cut(df[col], bins=bins, labels=labels, right=False).value_counts().sort_index()
        result.loc[col] = counts

    return result.fillna(0).astype(int)


In [58]:
liquidity_bucket_table(df[['liq_CE_0', 'liq_PE_0', 'liq_CE_5', 'liq_PE_5', 'liq_PE_10', 'liq_CE_10']])

,1-10,11-20,21-30,31-40,41-50,51-60,61-70,71-80,81-90,91-100
liq_CE_0,0,0,0,0,0,0,0,2,6,236
liq_PE_0,0,0,0,0,0,0,0,1,4,239
liq_CE_5,6,4,6,4,3,6,16,21,49,129
liq_PE_5,4,6,3,3,8,8,9,26,65,111
liq_PE_10,26,19,13,13,22,20,26,21,13,25
liq_CE_10,22,17,12,13,7,24,21,25,22,62


In [1]:
import pandas as pd
import numpy as np

def liquidity_bucket_table_pct(df, step=10, min_val=1, max_val=100):
    # Define bin edges and labels
    bins = list(range(min_val, max_val + 1, step))
    bins.append(max_val + 1)  # to include the final upper bound
    labels = [f"{i}-{i+step-1}" for i in range(min_val, max_val + 1, step)]

    # Initialize result DataFrame
    result = pd.DataFrame(index=df.columns, columns=labels)

    # Count and normalize values in each bin for each column
    for col in df.columns:
        binned = pd.cut(df[col], bins=bins, labels=labels, right=False)
        counts = binned.value_counts(normalize=True).sort_index() * 100  # convert to %
        result.loc[col] = counts

    result1 = result.fillna(0).round(2)
    result = result1['91-100'].mean().round(2)

    # return result.fillna(0).round(2)  # round to 2 decimals
    return result, result1


In [92]:
# df_2 = liquidity_bucket_table_pct(df[['liq_CE_0', 'liq_PE_0', 'liq_CE_5', 'liq_PE_5', 'liq_PE_10', 'liq_CE_10']])

In [2]:
import glob
file_list = glob.glob("/home/cloudcraftz/Downloads/data_reports/2024/*.csv")
len(file_list)

89

In [3]:
import os
os.makedirs("/home/cloudcraftz/Downloads/data_reports/2024/liquadity_5_pct_otm", exist_ok=True)

In [4]:

final_result = {}
for file in file_list:
    df = pd.read_csv(file, parse_dates=["Date"], index_col="Date")
    result = liquidity_bucket_table_pct(df[['liq_CE_0', 'liq_PE_0']])

    result[-1].to_csv(os.path.join("/home/cloudcraftz/Downloads/data_reports/2024/liquadity", file.split("/")[-1]))

    final_result[file.split("/")[-1].split(".")[0]] = result[0]
    # print(file.split("/")[-1].split(".")[0])

df_1 = pd.DataFrame(list(final_result.items()), columns=['Underlying', 'Value'])


In [6]:
# Plot line chart
# import plotly.express as px

# fig = px.line(df_1, x='Underlying', y='Value', title='Liquidity Line Chart', markers=True)
# fig.update_traces(line=dict(color='blue'), marker=dict(size=8))
# fig.update_layout(xaxis_title='Underlying', yaxis_title='Value')
# fig.show()

In [6]:
import plotly.express as px
df_sorted = df_1.sort_values(by='Value', ascending=False)

# Plot
fig = px.bar(df_sorted, x='Underlying', y='Value', text_auto=True, title='Liquidity Bar Chart ATM Option')
fig.update_layout(xaxis_title='Underlying', yaxis_title='Liquidity')
fig.show()

In [1]:
len(df_1[df_1['Value'] >= 95]['Underlying'].to_list())

NameError: name 'df_1' is not defined

In [8]:

final_result = {}
for file in file_list:
    df = pd.read_csv(file, parse_dates=["Date"], index_col="Date")
    result = liquidity_bucket_table_pct(df[['liq_CE_5', 'liq_PE_5']])

    result[-1].to_csv(os.path.join("/home/cloudcraftz/Downloads/data_reports/2024/liquadity_5_pct_otm", file.split("/")[-1]))

    final_result[file.split("/")[-1].split(".")[0]] = result[0]
    # print(file.split("/")[-1].split(".")[0])

df_2 = pd.DataFrame(list(final_result.items()), columns=['Underlying', 'Value'])


In [9]:
df_sorted_2 = df_2.sort_values(by='Value', ascending=False)

# Plot
fig = px.bar(df_sorted_2, x='Underlying', y='Value', text_auto=True, title='Liquidity Bar Chart 5 % OTM Option')
fig.update_layout(xaxis_title='Underlying', yaxis_title='Liquidity')
fig.show()

In [7]:
df_sumegh = pd.read_csv("/home/cloudcraftz/Downloads/tiger_test/sumegh_test.txt", header=1)

In [9]:
df_sumegh

,20240101_BOSCHLTD_HistoricData.txt
0,20240101_BRITANNIA_HistoricData.txt
1,20240101_CANFINHOME_HistoricData.txt
2,20240101_CHOLAFIN_HistoricData.txt
3,20240101_CIPLA_HistoricData.txt
4,20240101_COLPAL_HistoricData.txt
...,...
2228,20250227_NIFTY_HistoricData.txt
2229,20250228_BANKNIFTY_HistoricData.txt
2230,20250228_NIFTY_HistoricData.txt
2231,20250303_BANKNIFTY_HistoricData.txt


In [11]:
df_sumegh = pd.read_csv("/home/cloudcraftz/Downloads/tiger_test/sumegh_test.txt", header=1)
df_sumegh['underlying'] = df_sumegh['20240101_BOSCHLTD_HistoricData.txt'].apply(lambda x: x.split("_")[1])
df_sumegh['underlying'].unique()

In [13]:
df_sumegh['underlying'].unique()

array(['BRITANNIA', 'CANFINHOME', 'CHOLAFIN', 'CIPLA', 'COLPAL',
       'CROMPTON', 'BOSCHLTD', 'BANKNIFTY', 'NIFTY'], dtype=object)

In [14]:
df = pd.read_csv("/home/cloudcraftz/Downloads/SingleStockDerivatives(Sheet1).csv")
df[(df['STATUS']=='Completed')&(df['IN PRIORITY LIST']==True)][['SYMBOL']]

In [25]:
file_path = glob.glob("/home/cloudcraftz/Downloads/data_reports/2024/liquadity/*.csv")

data_list = []
for file in file_path:
    data_list.append(file.split("/")[-1].split(".")[0])

In [27]:
data_list = set(data_list)

In [32]:
data_to_download = []
for symbol in df[(df['STATUS']=='Completed')&(df['IN PRIORITY LIST']==True)]['SYMBOL']:
    if symbol not in data_list:
        data_to_download.append(symbol)

In [35]:
data_to_download

['BOSCHLTD',
 'LALPATHLAB',
 'HDFCBANK',
 'HDFCLIFE',
 'HEROMOTOCO',
 'HINDUNILVR',
 'ICICIBANK',
 'ICICIGI',
 'IDFCFIRSTB',
 'ITC',
 'INDUSTOWER',
 'INDUSINDBK',
 'NAUKRI',
 'INFY',
 'INDIGO',
 'JUBLFOOD',
 'KOTAKBANK',
 'LTF',
 'LTTS',
 'LICHSGFIN',
 'LTIM',
 'LT',
 'MRF',
 'M&MFIN',
 'M&M',
 'MANAPPURAM',
 'MARICO',
 'MARUTI',
 'MPHASIS',
 'MCX',
 'MUTHOOTFIN',
 'NAVINFLUOR',
 'NESTLEIND',
 'PIIND',
 'PAGEIND',
 'PETRONET',
 'PIDILITIND',
 'POLYCAB',
 'POWERGRID',
 'RECLTD',
 'RELIANCE',
 'SBICARD',
 'SBILIFE',
 'SHREECEM',
 'SRF',
 'SHRIRAMFIN',
 'SIEMENS',
 'SUNPHARMA',
 'TATACONSUM',
 'TVSMOTOR',
 'TCS',
 'TATAMOTORS',
 'TECHM',
 'INDHOTEL',
 'TITAN',
 'TRENT',
 'ULTRACEMCO',
 'UBL',
 'UNITDSPR',
 'VOLTAS']

In [39]:
aa = """ABB,
ABBOTINDIA,
ABCAPITAL,
APOLLOHOSP,
ASHOKLEY,
ASIANPAINT,
ASTRAL,
AUBANK,
AXISBANK,
BAJAJ-AUTO,
BAJFINANCE,
BALKRISIND,
BERGEPAINT,
BHARTIARTL,
BIOCON,
BRITANNIA,
CANFINHOME,
CHOLAFIN,
CIPLA,
COLPAL,
CROMPTON,
CUMMINSIND,
DABUR,
DIVISLAB,
EICHERMOT,
ESCORTS,
FEDERALBNK,
GODREJCP,
GUJGASLTD,
HAVELLS,
HCLTECH,
HDFCAMC,
HDFCBANK,
HDFCLIFE,
HEROMOTOCO,
HINDUNILVR,
ICICIBANK,
ICICIGI,
ICICIPRULI,
IDFCFIRSTB,
INDHOTEL,
INDUSINDBK,
ITC,
LALPATHLAB,
SBIN
"""

In [41]:
onedrivelist = aa.split(",\n")

In [42]:
need_run = []
for i in onedrivelist:
    if i in data_to_download:
        need_run.append(i)

In [46]:
need_run

11

In [ ]:
"""ITC,


"""

In [68]:
liquidity_bucket_table_pct(df[['liq_CE_0', 'liq_PE_0']])

          1-10  11-20  21-30  31-40  41-50  51-60  61-70  71-80  81-90  91-100
liq_CE_0   0.0    0.0    0.0    0.0    0.0    0.0    0.0   0.82   2.46   96.72
liq_PE_0   0.0    0.0    0.0    0.0    0.0    0.0    0.0   0.41   1.64   97.95
97.33500000000001


AttributeError: 'numpy.float64' object has no attribute 'fillna'

In [64]:
df_2.to_csv("liquidity_bucket_table.csv")

In [65]:
col_name_list = ['spread_CE_0_max_time','spread_CE_5_max_time','spread_CE_-5_max_time',
            'spread_PE_0_max_time','spread_PE_5_max_time','spread_PE_-5_max_time']

for col_name in col_name_list:
    spread_time_plot(col_name, title=f'Time of Max Spread | {col_name}')

In [25]:
col_name_list = ['spread_CE_0_min_time','spread_CE_5_min_time','spread_CE_-5_min_time',
            'spread_PE_0_min_time','spread_PE_5_min_time','spread_PE_-5_min_time']

for col_name in col_name_list:
    spread_time_plot(col_name, title=f'Time of Min Spread | {col_name}')